# Classification heatmaps: MCC and ROC-AUC by ee threshold

The format is the same as in `heatmaps_MAE_AUC90.ipynb`: horizontal panels in the order
**Random → Catalyst (GroupKFold-10) → LORGO**, with each cell showing a point estimate and, below it, ± half the width of the 95%
confidence interval (cluster bootstrap from the classification `metrics_summary.csv`).

Figures, one per threshold:
1. `heatmap_classification_MCC_ee70`, `..._ee80`, `..._ee90` — Matthews correlation coefficient (MCC);
2. `heatmap_classification_AUC_ee70`, `..._ee80`, `..._ee90` — ROC-AUC.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ----------------------------- CONFIGURATION -----------------------------
CLF_CSV = Path("../results/classification/metrics_summary.csv")
OUT_DIR = Path("../results/figures/figures_heatmaps_clf")

THRESHOLDS = [70, 80, 90]
METRICS = {"MCC": ("MCC", 0.0), "ROC_AUC": ("ROC-AUC", 0.5)}   # column: (label, scale center)
COMMON_SCALE = True      # one color scale for all thresholds of a given metric
DPI_PNG = 600

FIG_WIDTH_MM  = 180
FIG_HEIGHT_MM = 125
FS_VALUE = 7
FS_CI    = 5.5
FS_TICK  = 7
FS_TITLE = 7.5
CMAP = "RdBu"
# -------------------------------------------------------------------------

SPLITS = ["random", "gkf10_catalyst", "lorgo_group"]
SPLIT_LABELS = {"random": "Random (5-fold × 5)",
                "gkf10_catalyst": "GroupKFold-10 (catalyst)",
                "lorgo_group": "LORGO (research group)"}
REP_ORDER = ["physchem", "maccs", "ecfp4", "ecfp4_count", "drfp", "onehot", "dft", "cgr"]
REP_LABELS = {"physchem": "Physchem", "maccs": "MACCS", "ecfp4": "ECFP4", "ecfp4_count": "ECFP4-count",
              "drfp": "DRFP", "onehot": "OHE", "dft": "DFT", "cgr": "CGR + catalyst"}
CLF_MODELS = ["logreg", "svc", "rf", "xgb", "chemprop"]
MODEL_LABELS = {"logreg": "LogReg", "svc": "SVC", "rf": "RF", "xgb": "XGBoost", "chemprop": "D-MPNN"}

MM = 1 / 25.4
MISSING = "#f0f0f0"
TEXT_DARK = "#222222"
OUT_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42, "svg.fonttype": "none",
})

clf = pd.read_csv(CLF_CSV)
clf = clf[clf.model != "dummy"]
print("classification:", clf.shape, "| thresholds:", sorted(clf.threshold.unique()))

## Helper functions

In [ ]:
def grid(df, col, models):
    """Representation × model matrix for a single column."""
    return (df.pivot(index="representation", columns="model", values=col)
              .reindex(index=REP_ORDER, columns=models).to_numpy(float))


def heat(ax, df, metric, models, cmap, norm, title, ylabels=True):
    M = grid(df, metric, models)
    half = (grid(df, f"{metric}_ci_high", models) - grid(df, f"{metric}_ci_low", models)) / 2
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, norm=norm, aspect="auto")
    ax.set_facecolor(MISSING)
    for (i, j), v in np.ndenumerate(M):
        if np.isnan(v):
            continue
        r, g, b, _ = im.cmap(im.norm(v))
        col = "white" if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5 else TEXT_DARK
        txt = f"{v:.2f}".replace("-0.00", "0.00")       # no negative zero
        ax.text(j, i - 0.14, txt, ha="center", va="center", fontsize=FS_VALUE, color=col)
        ax.text(j, i + 0.22, f"± {half[i, j]:.2f}", ha="center", va="center", fontsize=FS_CI, color=col)
    ax.set_xticks(range(len(models)), [MODEL_LABELS[m] for m in models], rotation=40, ha="right",
                  rotation_mode="anchor")
    ax.set_yticks(range(len(REP_ORDER)), [REP_LABELS[r] for r in REP_ORDER] if ylabels else [])
    ax.tick_params(length=0, labelsize=FS_TICK, pad=2)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, fontsize=FS_TITLE, pad=4)
    return im


def colorbar(fig, im, ax, label=None, shrink=1.0, aspect=30):
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.045, pad=0.02, aspect=aspect, shrink=shrink)
    cb.ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
    cb.ax.tick_params(labelsize=FS_TICK - 1, length=2, pad=1)
    cb.outline.set_visible(False)
    if label:
        cb.set_label(label, fontsize=FS_TICK)
    return cb


def save(fig, stem):
    for ext in ("pdf", "svg"):
        fig.savefig(OUT_DIR / f"{stem}.{ext}", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=DPI_PNG, bbox_inches="tight")


def norm_for(metric, center, df):
    half = float(np.nanmax(np.abs(df[metric] - center)))
    return TwoSlopeNorm(vcenter=center, vmin=center - half, vmax=center + half)


def figure(metric, threshold):
    label, center = METRICS[metric]
    sub_t = clf[clf.threshold == threshold]
    norm = norm_for(metric, center, clf if COMMON_SCALE else sub_t)
    fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH_MM * MM, FIG_HEIGHT_MM * MM), constrained_layout=True)
    for k, (ax, s) in enumerate(zip(axes, SPLITS)):
        im = heat(ax, sub_t[sub_t.split == s], metric, CLF_MODELS, CMAP, norm,
                  f"{SPLIT_LABELS[s]} — ee ≥ {threshold}%", ylabels=(k == 0))
    colorbar(fig, im, list(axes), label, shrink=0.5, aspect=40)
    stem = f"heatmap_classification_{'AUC' if metric == 'ROC_AUC' else metric}_ee{threshold}"
    save(fig, stem)
    print("saved", OUT_DIR / stem)
    plt.show()

## 1. MCC

In [ ]:
for t in THRESHOLDS:
    figure("MCC", t)

## 2. ROC-AUC

In [ ]:
for t in THRESHOLDS:
    figure("ROC_AUC", t)